# Oracle Agent Memory — OAMP vs Naive Memory Benchmarks (OCI)

[![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)

This notebook quantifies the practical benefits of Oracle Agent Memory (OAMP) over a naive flat-history memory approach along three axes:

1. **Token consumption** per turn — cost
2. **Wall-clock latency** per turn — speed, both retrieval-only and end-to-end
3. **Response quality** — accuracy, via LLM-as-a-judge

The OCI variant runs generation, judging, summarisation, extraction, and embeddings through OCI Generative AI. It defaults to a smaller interactive run through `MAX_BENCHMARK_TURNS`; set that environment variable to `80` for the full scripted benchmark.

> **Prerequisites**
> - An Oracle AI Database instance reachable via `DB_CONNECT_STRING`
> - OCI Generative AI access to `xai.grok-3-fast`
> - Local OCI config file at `~/.oci/config`, or set `OCI_CONFIG_FILE` and `OCI_PROFILE`
> - The `oracleagentmemory` package installed in the active kernel environment


## 1. Install dependencies

In [ ]:
%pip install --upgrade pip
%pip install oracleagentmemory==26.4.0 oci matplotlib numpy nest_asyncio


In [ ]:
import configparser
import os
from pathlib import Path


def _expand_path(value: str) -> str:
    return os.path.expanduser(os.path.expandvars(value))


def _read_oci_profile_region(config_file: str, profile_name: str) -> str:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return ""

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT" and parser.defaults().get("region"):
        return parser.defaults()["region"].strip()
    for section in (profile_name, f"profile {profile_name}"):
        if parser.has_section(section) and parser.has_option(section, "region"):
            return parser.get(section, "region").strip()
    return ""


def _read_oci_profile_values(config_file: str, profile_name: str) -> dict[str, str]:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return {}

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT":
        values = dict(parser.defaults())
    else:
        values = {}
        for section in (profile_name, f"profile {profile_name}"):
            if parser.has_section(section):
                values.update(dict(parser.items(section)))
                break

    if values.get("key_file"):
        values["key_file"] = _expand_path(values["key_file"])
    return values


def _set_oci_env_aliases(config_file: str, profile_name: str) -> None:
    values = _read_oci_profile_values(config_file, profile_name)
    aliases = {
        "user": ("OCI_USER", "OCI_USER_ID"),
        "tenancy": ("OCI_TENANCY", "OCI_TENANCY_ID"),
        "fingerprint": ("OCI_FINGERPRINT",),
        "key_file": ("OCI_KEY_FILE",),
        "region": ("OCI_REGION", "OCI_REGION_NAME"),
    }
    for config_key, env_names in aliases.items():
        value = values.get(config_key, "")
        if not value:
            continue
        for env_name in env_names:
            os.environ.setdefault(env_name, value)


def _env(name: str, default: str = "") -> str:
    raw_value = os.environ.get(name)
    value = raw_value if raw_value not in (None, "") else default
    value = _expand_path(value) if name.endswith("_FILE") else value
    os.environ[name] = value
    return value


def _require_non_empty(value: str, name: str, hint: str) -> None:
    if not value:
        raise RuntimeError(f"Missing {name}. {hint}")


def _require_existing_file(value: str, name: str, hint: str) -> None:
    _require_non_empty(value, name, hint)
    if not Path(value).expanduser().exists():
        raise RuntimeError(f"{name} does not exist at {value}. {hint}")


LLM_PROVIDER = _env("LLM_PROVIDER", "oci_genai_native").strip().lower().replace("-", "_")
if LLM_PROVIDER not in {"oci_genai_native", "oci_native"}:
    raise RuntimeError("This OCI notebook supports LLM_PROVIDER=oci_genai_native only.")

LLM_MODEL = _env("LLM_MODEL", "xai.grok-3-fast").strip()
OCI_CONFIG_FILE = _env("OCI_CONFIG_FILE", "~/.oci/config")
OCI_PROFILE = _env("OCI_PROFILE", "DEFAULT")
OCI_PROFILE_REGION = _read_oci_profile_region(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_REGION = os.environ.get("OCI_GENAI_REGION", "").strip() or "us-chicago-1"
os.environ["OCI_GENAI_REGION"] = OCI_GENAI_REGION
OCI_GENAI_ENDPOINT = _env(
    "OCI_GENAI_ENDPOINT",
    f"https://inference.generativeai.{OCI_GENAI_REGION}.oci.oraclecloud.com",
)
OCI_PROFILE_VALUES = _read_oci_profile_values(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_COMPARTMENT_OCID = _env(
    "OCI_GENAI_COMPARTMENT_OCID",
    OCI_PROFILE_VALUES.get("compartment_id") or OCI_PROFILE_VALUES.get("tenancy", ""),
).strip()
OCI_GENAI_MODEL_OCID = _env("OCI_GENAI_MODEL_OCID", LLM_MODEL).strip()
OCI_GENAI_EMBED_MODEL = _env("OCI_GENAI_EMBED_MODEL", "cohere.embed-english-v3.0")

_set_oci_env_aliases(OCI_CONFIG_FILE, OCI_PROFILE)
os.environ.setdefault("OCI_REGION", OCI_GENAI_REGION)
os.environ.setdefault("OCI_REGION_NAME", OCI_GENAI_REGION)
os.environ.setdefault("OCI_COMPARTMENT_ID", OCI_GENAI_COMPARTMENT_OCID)

os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

_require_existing_file(
    OCI_CONFIG_FILE,
    "OCI_CONFIG_FILE",
    "Set OCI_CONFIG_FILE to your OCI config path, usually ~/.oci/config.",
)
_require_non_empty(OCI_PROFILE, "OCI_PROFILE", "Set OCI_PROFILE to a profile in OCI_CONFIG_FILE.")
_require_non_empty(OCI_GENAI_REGION, "OCI_GENAI_REGION", "Set OCI_GENAI_REGION for OCI Generative AI.")
_require_non_empty(
    OCI_GENAI_COMPARTMENT_OCID,
    "OCI_GENAI_COMPARTMENT_OCID",
    "Set this to a compartment OCID with OCI Generative AI permissions. The notebook falls back to compartment_id or tenancy from OCI_CONFIG_FILE when present.",
)
_require_non_empty(
    OCI_GENAI_MODEL_OCID,
    "OCI_GENAI_MODEL_OCID",
    "Set this to the on-demand model OCID or model ID. Defaults to LLM_MODEL, for example xai.grok-3-fast.",
)

import nest_asyncio
nest_asyncio.apply()

print("Environment configured for OCI Generative AI.")
print(f"LLM provider: {LLM_PROVIDER}")
print(f"LLM model: {LLM_MODEL}")
print(f"OCI profile region: {OCI_PROFILE_REGION or 'not set'}")
print(f"OCI GenAI region: {OCI_GENAI_REGION}")
print(f"OCI endpoint: {OCI_GENAI_ENDPOINT}")
print(f"OCI compartment: {OCI_GENAI_COMPARTMENT_OCID[:18]}..." if OCI_GENAI_COMPARTMENT_OCID else "OCI compartment: not set")
print(f"OCI model id: {OCI_GENAI_MODEL_OCID}")
print(f"OCI embedding model: {OCI_GENAI_EMBED_MODEL}")


import asyncio
import json
from collections.abc import Sequence
from typing import Any

import numpy as np
import oci
from oracleagentmemory.apis.embedders.embedder import IEmbedder
from oracleagentmemory.apis.llms.llm import ILlm, LlmResponse


class OciSdkEmbedder(IEmbedder):
    """OracleAgentMemory embedder backed directly by OCI Generative AI."""

    def __init__(self, client, compartment_id: str, model_id: str):
        self.client = client
        self.compartment_id = compartment_id
        self.model_id = model_id

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        from oci.generative_ai_inference import models

        details = models.EmbedTextDetails()
        details.inputs = texts
        details.compartment_id = self.compartment_id
        details.serving_mode = models.OnDemandServingMode(model_id=self.model_id)
        details.truncate = models.EmbedTextDetails.TRUNCATE_END
        details.input_type = (
            models.EmbedTextDetails.INPUT_TYPE_SEARCH_QUERY
            if is_query
            else models.EmbedTextDetails.INPUT_TYPE_SEARCH_DOCUMENT
        )

        response = self.client.embed_text(details)
        embeddings = getattr(response.data, "embeddings", None)
        if not embeddings:
            raise RuntimeError("OCI embed_text returned no embeddings.")
        return np.asarray(embeddings, dtype=np.float32)

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


def create_oci_genai_client():
    try:
        oci_config = oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
    except Exception as exc:
        raise RuntimeError(
            f"Could not load OCI config profile {OCI_PROFILE!r} from {OCI_CONFIG_FILE}: {exc}"
        ) from exc

    return oci.generative_ai_inference.GenerativeAiInferenceClient(
        config=oci_config,
        service_endpoint=OCI_GENAI_ENDPOINT,
        retry_strategy=oci.retry.NoneRetryStrategy(),
        timeout=(10, 240),
    )


def _message_content_to_text(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text") or item.get("content") or ""))
            else:
                parts.append(str(getattr(item, "text", "") or getattr(item, "content", "") or item))
        return "\n".join(part for part in parts if part)
    return str(content)


def _oci_text_content(models, text: Any):
    item = models.TextContent()
    if hasattr(models.TextContent, "TYPE_TEXT"):
        item.type = models.TextContent.TYPE_TEXT
    item.text = _message_content_to_text(text)
    return item


def _chat_message_to_oci(models, message: dict):
    role = (message.get("role") or "user").lower()
    content_items = [_oci_text_content(models, message.get("content"))]
    if role == "system":
        oci_message = models.SystemMessage()
        oci_message.role = models.SystemMessage.ROLE_SYSTEM
    elif role == "assistant":
        oci_message = models.AssistantMessage()
        oci_message.role = models.AssistantMessage.ROLE_ASSISTANT
    else:
        oci_message = models.UserMessage()
        oci_message.role = models.UserMessage.ROLE_USER
    oci_message.content = content_items
    return oci_message


def _extract_oci_text(message: Any) -> str:
    parts = []
    for item in getattr(message, "content", None) or []:
        text = getattr(item, "text", None)
        if text:
            parts.append(text)
    return "\n".join(parts)


def oci_chat_text(
    messages: list[dict[str, str]],
    *,
    temperature: float = 0.2,
    max_tokens: int = 1200,
    response_json_schema: dict[str, Any] | None = None,
) -> str:
    from oci.generative_ai_inference import models

    chat_request = models.GenericChatRequest()
    chat_request.api_format = models.BaseChatRequest.API_FORMAT_GENERIC
    chat_request.messages = [_chat_message_to_oci(models, message) for message in messages]
    chat_request.max_tokens = max_tokens
    chat_request.temperature = temperature
    chat_request.top_p = 1.0
    chat_request.top_k = 0
    chat_request.is_stream = False

    if response_json_schema:
        schema = models.ResponseJsonSchema()
        schema.name = "response"
        schema.schema = response_json_schema
        schema.is_strict = False
        response_format = models.JsonSchemaResponseFormat()
        response_format.type = models.JsonSchemaResponseFormat.TYPE_JSON_SCHEMA
        response_format.json_schema = schema
        chat_request.response_format = response_format

    serving_mode = models.OnDemandServingMode()
    serving_mode.model_id = OCI_GENAI_MODEL_OCID

    chat_detail = models.ChatDetails()
    chat_detail.serving_mode = serving_mode
    chat_detail.chat_request = chat_request
    chat_detail.compartment_id = OCI_GENAI_COMPARTMENT_OCID

    response = genai_client.chat(chat_detail)
    data = response.data
    chat_response = getattr(data, "chat_response", data)
    choices = getattr(chat_response, "choices", None) or []
    message = getattr(choices[0], "message", None) if choices else None
    return _extract_oci_text(message)


class OciSdkLlm(ILlm):
    """OracleAgentMemory LLM backed directly by OCI Generative AI."""

    def __init__(self, temperature: float = 0.2, max_tokens: int = 1200):
        self.temperature = temperature
        self.max_tokens = max_tokens

    def generate(
        self,
        prompt: str | Sequence[dict[str, str]],
        *,
        response_json_schema: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> LlmResponse:
        if isinstance(prompt, str):
            messages = [{"role": "user", "content": prompt}]
        else:
            messages = list(prompt)
        text = oci_chat_text(
            messages,
            temperature=kwargs.get("temperature", self.temperature),
            max_tokens=kwargs.get("max_tokens", self.max_tokens),
            response_json_schema=response_json_schema,
        )
        return LlmResponse(text=text)

    async def generate_async(
        self,
        prompt: str | Sequence[dict[str, str]],
        *,
        response_json_schema: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> LlmResponse:
        return await asyncio.to_thread(
            self.generate,
            prompt,
            response_json_schema=response_json_schema,
            **kwargs,
        )


genai_client = create_oci_genai_client()
oamp_embedder = OciSdkEmbedder(
    genai_client,
    compartment_id=OCI_GENAI_COMPARTMENT_OCID,
    model_id=OCI_GENAI_EMBED_MODEL,
)
oamp_llm = OciSdkLlm(temperature=0.2)
print("OCI Generative AI helpers ready.")

MAX_BENCHMARK_TURNS = int(os.environ.get("MAX_BENCHMARK_TURNS", "20"))
print(f"MAX_BENCHMARK_TURNS: {MAX_BENCHMARK_TURNS} (set to 80 for the full run)")


## 2. Connect to Oracle and create an OAMP client

We attach `OciSdkLlm` so OAMP can automatically extract memories from every user/assistant message written to the thread. The extraction config is tuned for frequent updates so the context card stays fresh throughout the run.


In [ ]:
import oracledb
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.apis.thread import Message

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)

client = OracleAgentMemory(
    connection=connection,
    embedder=oamp_embedder,
    llm=oamp_llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
)

USER_ID = "benchmark-user"
AGENT_ID = "benchmark-agent"

for create_fn, eid, info in [
    (client.add_user, USER_ID, "Richmond - benchmarking OAMP vs naive memory."),
    (client.add_agent, AGENT_ID, "OAMP-backed research assistant for benchmarks."),
]:
    try:
        create_fn(eid, info)
    except ValueError as exc:
        if "already exists" not in str(exc):
            raise

thread = client.create_thread(
    user_id=USER_ID,
    agent_id=AGENT_ID,
    enable_context_summary=True,
    context_summary_update_frequency=2,
    memory_extraction_frequency=2,
    memory_extraction_window=4,
)
print(f"OAMP thread created: {thread.thread_id}")


## 3. Define both agents with instrumentation

Both agents call `xai.grok-3-fast` through the native OCI helper — no agent framework — so we can observe the exact `messages` list sent on every turn. Four streams of data are captured per turn:

- **Token count** — for the cost plot
- **Retrieval latency** — prompt-assembly time
- **End-to-end latency** — query received to answer ready
- **Response text** — so the LLM judge can evaluate accuracy


In [ ]:
import json as _json_lib
import time
import matplotlib.pyplot as plt

SYSTEM_PROMPT = "You are a concise research assistant. Answer in 1-3 sentences."


def estimate_tokens(messages: list) -> int:
    """Approximate input tokens as chars/4."""
    return len(_json_lib.dumps(messages)) // 4


def call_chat(messages: list[dict[str, str]], *, temperature: float = 0.2, max_tokens: int = 1200) -> str:
    return oci_chat_text(messages, temperature=temperature, max_tokens=max_tokens)


def align_metric_series(label: str, *series):
    """Return x-axis turns plus equally sized series for partial or rerun-safe plots."""
    lengths = [len(values) for values in series]
    if not lengths or min(lengths) == 0:
        raise RuntimeError(f"No data to plot for {label}. Run the benchmark cells that populate the metric arrays first.")
    n = min(lengths)
    if len(set(lengths)) != 1:
        print(f"[warn] {label}: aligning series to the first {n} turns; observed lengths={lengths}.")
        print("       Restart the kernel and rerun from the setup cells for a clean full comparison.")
    return list(range(1, n + 1)), [list(values)[:n] for values in series]


# Per-turn metrics for plotting
oamp_token_history = []
naive_token_history = []

oamp_retrieval_latency = []
oamp_total_latency = []
naive_retrieval_latency = []
naive_total_latency = []

oamp_responses = []
naive_responses = []

naive_messages = [{"role": "system", "content": SYSTEM_PROMPT}]


def call_oamp_agent(user_query: str) -> str:
    """OAMP agent: prompt = system + retrieved context card + latest user message."""
    t_start = time.perf_counter()
    thread.add_messages([Message(role="user", content=user_query)])
    context_card = thread.get_context_card() or "(no prior context)"
    t_context_built = time.perf_counter()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Relevant memory:\n{context_card}\n\nCurrent question: {user_query}"},
    ]
    oamp_token_history.append(estimate_tokens(messages))

    answer = call_chat(messages)
    thread.add_messages([Message(role="assistant", content=answer)])

    t_end = time.perf_counter()
    oamp_retrieval_latency.append(t_context_built - t_start)
    oamp_total_latency.append(t_end - t_start)
    oamp_responses.append(answer)
    return answer


def call_naive_agent(user_query: str) -> str:
    """Naive agent: prompt = full accumulated history + latest user message."""
    t_start = time.perf_counter()

    naive_messages.append({"role": "user", "content": user_query})
    t_context_built = time.perf_counter()
    naive_token_history.append(estimate_tokens(naive_messages))

    answer = call_chat(naive_messages)
    naive_messages.append({"role": "assistant", "content": answer})

    t_end = time.perf_counter()
    naive_retrieval_latency.append(t_context_built - t_start)
    naive_total_latency.append(t_end - t_start)
    naive_responses.append(answer)
    return answer


print("Agents ready. Tracking tokens, retrieval latency, end-to-end latency, and responses.")


## 4. Run the scripted conversation

The conversation contains dense declarative turns about ChromAtlas-ND followed by recall turns that force retrieval.

> **Warning:** the full run makes many OCI Generative AI calls. The notebook defaults to `MAX_BENCHMARK_TURNS=20` for interactive use. Set `MAX_BENCHMARK_TURNS=80` before running the config cell for the full reproduction.


In [ ]:
conversation_turns = [
    # ---------- Block 1: declarative core (turns 1-22) ----------
    (
        "Hi! I'm Dr. Richmond Alake, a senior computational genomics researcher at the Oracle Life Sciences Institute. "
        "I lead a team of eight bioinformaticians investigating non-coding regulatory variants in rare pediatric "
        "neurodevelopmental disorders. Most of our cohort is from the GeneDx Trio Consortium, and our grant is funded "
        "by the NIH Common Fund's 4D Nucleome program through 2028."
    ),
    (
        "The specific project I want help with is called ChromAtlas-ND. We are building a whole-genome variant "
        "annotation pipeline that integrates long-read Oxford Nanopore PromethION data with short-read Illumina "
        "NovaSeq X data across 3,412 trios. The goal is to phase de novo structural variants and link them to "
        "cell-type-specific enhancer activity in fetal cortical neurons."
    ),
    (
        "My principal wet-lab collaborator is Dr. Sarah Chen at Baylor College of Medicine in Houston. Sarah runs "
        "the single-cell ATAC-seq and Hi-C arm — she is generating 10x Multiome data on matched iPSC-derived "
        "cortical organoids from 60 probands. She holds a joint appointment with the Jan and Dan Duncan Neurological "
        "Research Institute and publishes heavily on 3D chromatin architecture."
    ),
    (
        "On the computational side, we are comparing three variant-prioritization strategies. The first is a pure "
        "sequence-based approach using Enformer embeddings of 200kb windows around each variant. The second is a "
        "hybrid strategy combining Enformer features with Hi-C contact maps, PhyloP conservation scores, and GTEx "
        "eQTL effect sizes inside an XGBoost ensemble. The third is a graph-based approach over a knowledge graph "
        "linking variants, genes, regulatory elements, and phenotypes using HPO ontology terms."
    ),
    (
        "I personally prefer the hybrid XGBoost approach because it lets us fuse semantic features from Enformer with "
        "structured biological priors like conservation and 3D contacts. In our preliminary benchmarks on the ClinVar "
        "pathogenic-vs-benign holdout, the hybrid model gets AUROC 0.91 compared to 0.84 for pure Enformer and 0.78 "
        "for the graph-only approach. The interpretability from SHAP values on the structured features is also "
        "something our clinical geneticists actually trust."
    ),
    (
        "We also have an internal agent called GenomeBot that sits on top of our Oracle Autonomous Database. It "
        "handles variant-lookup tickets from the clinical team, runs VEP and SpliceAI on ad-hoc VCFs, and posts "
        "prioritized candidates back into our Epic EHR integration. It currently serves about 200 clinicians across "
        "Texas Children's Hospital and the Oracle Health network."
    ),
    (
        "Sarah just told me she is adding a new modality to the benchmark — CUT&Tag data for H3K27ac and H3K4me1 on "
        "the same 60 organoid lines, generated on an Illumina NextSeq 2000 at 30 million reads per sample. Please "
        "note that this will roughly double the feature space for the hybrid model and will require us to retrain "
        "with a leave-one-donor-out cross-validation scheme to avoid donor leakage."
    ),
    (
        "Our evaluation deadline is end of Q2 2026 because we have to present final results at the American Society "
        "of Human Genetics annual meeting in San Diego in October 2026, and the manuscript targeting Nature Genetics "
        "needs to be submitted by July. The grant renewal also depends on showing concrete clinical utility metrics, "
        "specifically the number of previously unsolved cases that received a likely-diagnostic non-coding variant "
        "through the pipeline."
    ),
    (
        "My second collaborator is Dr. Javier Morales at the Broad Institute of MIT and Harvard. Javier is a machine "
        "learning researcher specializing in interpretability methods for genomic deep learning models. He is leading "
        "the effort to adapt integrated gradients and attention rollout techniques so we can attribute Enformer "
        "predictions back to specific transcription factor binding motifs and chromatin state annotations."
    ),
    (
        "For reference datasets, we are using gnomAD v4.1 for allele frequency filtering, the UK Biobank 500K WGS "
        "release for population-scale burden testing, and the ENCODE4 rE2G map for enhancer-gene linking. We also "
        "pull in Roadmap Epigenomics chromatin states for the 127 reference tissues and PsychENCODE cell-type-"
        "specific regulatory maps for cortex, hippocampus, and cerebellum."
    ),
    (
        "A particular variant class we are prioritizing is tandem repeat expansions in enhancer regions — our pilot "
        "found 47 novel pathogenic TREs in introns and intergenic regions that were missed by every commercial "
        "diagnostic pipeline. We are using ExpansionHunter Denovo for discovery and Straglr for long-read "
        "confirmation, then validating with targeted Oxford Nanopore adaptive sampling on a Flongle."
    ),
    (
        "Our variant-calling preprocessing pipeline uses DeepVariant 1.9 for per-sample calling followed by GLnexus "
        "for joint genotyping across trios. Structural variants come from a consensus of Sniffles2, CuteSV, and "
        "Manta, merged with SURVIVOR. We VQSR-equivalent filter with hap.py against the Genome in a Bottle HG002 "
        "truth set and target an SNV F1 of at least 0.995."
    ),
    (
        "Compute is on a dedicated 4-rack Oracle Cloud Infrastructure GPU cluster with 128 NVIDIA A100 80GB GPUs, "
        "about 2.5 petabytes of ZFS-backed block storage, and a shared Lustre scratch tier. We schedule jobs through "
        "Nextflow on Slurm, and our Enformer fine-tuning runs take roughly 72 hours per epoch at mixed precision."
    ),
    (
        "Model training uses a federated learning setup coordinated across four academic medical centers — Baylor, "
        "Children's Hospital of Philadelphia, UCSF Benioff Children's, and Toronto SickKids. We use Flower as the "
        "federation framework with differentially private aggregation at epsilon=2.0 so no raw genotypes or "
        "phenotypes cross institutional boundaries."
    ),
    (
        "My third collaborator is Dr. Aisha Patel at Genomics England in Cambridge, UK. Aisha runs the polygenic "
        "risk score validation arm using the 100,000 Genomes Project rare-disease cohort and has access to linked "
        "NHS longitudinal phenotype data. Her group is helping us calibrate PRS distributions separately for the "
        "five 1000 Genomes super-populations to avoid ancestry bias in clinical deployment."
    ),
    (
        "Clinical validation is an 18-month prospective study on 850 currently-unsolved rare-disease cases drawn from "
        "the Undiagnosed Diseases Network. The primary endpoint is the diagnostic yield gain over the current GREEN "
        "gene panel standard. The secondary endpoint is time-to-diagnosis, measured from sample receipt to clinician "
        "sign-off of the report in Epic."
    ),
    (
        "Candidate variants flowing out of our pipeline feed into AlphaFold3 structural impact prediction. For "
        "missense variants we compute pLDDT shifts and interface-disruption scores against known protein-protein "
        "interactions from STRING v12. For splice-altering variants we run Pangolin and SpliceAI-32k ensembled with "
        "geometric mean, then cross-reference against MaxEntScan for donor/acceptor strength."
    ),
    (
        "We have a strategic partnership with Illumina, who is providing TruSight One Expanded capture kits at cost "
        "for the clinical validation cohort. The capture targets 6,794 clinically relevant genes plus 1.4 Mb of "
        "curated non-coding regulatory regions that our team nominated based on the first-round ChromAtlas-ND hits. "
        "Illumina's Andrew Kim is our technical point of contact."
    ),
    (
        "Publication plans are three papers. The main ChromAtlas-ND atlas paper targets Nature Genetics by July 2026. "
        "A companion methods paper on the hybrid XGBoost architecture and federated training targets Nature Methods "
        "by September. A clinical-impact paper reporting the 18-month diagnostic yield targets American Journal of "
        "Human Genetics in early 2027."
    ),
    (
        "Our funding stack combines a 5-year NIH R01 at 2.4M per year, 500K in Oracle for Research cloud credits "
        "renewed annually, a 1.8M catalytic grant from the Bill and Melinda Gates Foundation for the global-health "
        "ancestry calibration work, and a 350K supplement from the Chan Zuckerberg Initiative for open-source tool "
        "development. Our administrative PI on the Gates grant is Dr. Kemi Okafor at the Oracle Life Sciences Institute."
    ),
    (
        "An important constraint to remember: all raw sequencing data for the clinical validation cohort must remain "
        "inside the HIPAA-audited Oracle Cloud Ashburn region. Only derived features and de-identified model "
        "predictions can leave that enclave. This is why Javier's interpretability work at the Broad runs against "
        "synthetic test genomes rather than real patient data."
    ),
    (
        "One more operational detail: we hold a project-wide sync every Tuesday at 10am Central, with a monthly "
        "external steering committee review on the first Friday. The steering committee is chaired by Dr. Euan "
        "Ashley from Stanford and includes representatives from the NIH, GeneDx, and the patient advocacy group "
        "Global Genes. The next steering review is May 1, 2026."
    ),
    # ---------- Block 2: recall checks (turns 23-30) ----------
    (
        "Recall check 1 — what is the cohort size for ChromAtlas-ND, and which two sequencing platforms are we "
        "using for the primary variant calls?"
    ),
    (
        "Recall check 2 — please list my three main collaborators, their institutions, and what arm of the project "
        "each one leads."
    ),
    (
        "Recall check 3 — what are the three variant-prioritization strategies under comparison, and what were the "
        "ClinVar AUROC numbers for each? Which one do I prefer and why?"
    ),
    (
        "Recall check 4 — what new data modality did Sarah add, on what platform, at what read depth, and what "
        "cross-validation change did it force us to make?"
    ),
    (
        "Recall check 5 — describe GenomeBot: what does it do, where does it live, which EHR is it integrated "
        "with, and roughly how many clinicians use it?"
    ),
    (
        "Recall check 6 — what is our compute setup, what federation framework are we using for training, and "
        "what is our differential privacy epsilon?"
    ),
    (
        "Recall check 7 — what are our three planned publications, which journals are we targeting, and by what "
        "dates for each one? What is the primary endpoint of the 18-month clinical validation?"
    ),
    (
        "Final summary — give me a complete briefing on ChromAtlas-ND covering: cohort and sequencing, the three "
        "analysis strategies and their performance, all three collaborators and their roles, the GenomeBot "
        "production agent, the CUT&Tag addition, compute and federation setup, the Illumina partnership, all "
        "funding sources, publication plans, the clinical validation endpoint, and the HIPAA data-residency "
        "constraint."
    ),
    # ---------- Block 3: more declarative + recall (turns 31-50) ----------
    (
        "Update — we have a new collaborator: Dr. Mei Zhang at Johns Hopkins. Mei is a single-cell RNA-seq "
        "specialist joining for the cell-type annotation arm. Her group will integrate scRNA-seq from the same "
        "60 organoid lines with Sarah's ATAC-seq and Hi-C data to produce matched multi-modal embeddings."
    ),
    (
        "Specific patient case to remember: ND-2104, a 6-year-old with Christianson syndrome features. Our "
        "pipeline flagged a 14kb tandem repeat expansion in the SLC9A6 enhancer region, ~85kb upstream of the "
        "gene body. This is the first proband identified by ChromAtlas-ND that wasn't already in OMIM."
    ),
    (
        "Hardware update — we just added a new GPU pod with 32 NVIDIA H100s on OCI for the larger Enformer "
        "fine-tuning runs. This is in addition to our existing 128 A100 cluster. The H100 pod cuts our per-epoch "
        "training time from 72 hours to about 22 hours."
    ),
    (
        "Software stack pinning for reproducibility: Nextflow 24.10 as the primary workflow orchestrator, "
        "Snakemake 8.18 as a fallback for legacy components, MLflow for experiment tracking and model registry, "
        "and Weights & Biases for hyperparameter sweeps. All managed via micromamba environments on the cluster."
    ),
    (
        "Decision: we are switching from Manta to DELLY for structural variant calling in the trio context, after "
        "running an internal benchmark on the Genome in a Bottle HG002-HG004 trio. DELLY had better trio-aware "
        "filtering and lower false-positive rate on inversions specifically."
    ),
    (
        "Recall check — what variant did our pipeline identify in patient ND-2104, and what makes that case "
        "scientifically notable?"
    ),
    (
        "Reviewer feedback came back on our first Nature Genetics submission — three reviewers. R1 wants more "
        "diverse ancestry validation, especially African and Hispanic/Latino cohorts. R2 questions the privacy "
        "guarantees of our federated training setup and wants formal DP analysis. R3 was positive overall."
    ),
    (
        "New hire — Dr. Tomás Aguilar is joining as a postdoc starting March 2026. He's coming from Pompeu Fabra "
        "University in Barcelona where he worked on fine-mapping methods for GWAS loci. He'll lead our fine-mapping "
        "effort on the candidate non-coding variants."
    ),
    (
        "Compute cost note — we burned 47,000 OCI credits in March 2026 alone, the bulk of it on Enformer "
        "fine-tuning sweeps and Hi-C contact-map preprocessing. We need to be more careful about sweep budgets "
        "going forward; finance flagged it at the last review."
    ),
    (
        "Recall check — who is reviewer R2 in our Nature Genetics submission and what was their main concern?"
    ),
    (
        "We submitted an IRB amendment last week to add the Genomics England 100,000 Genomes Project rare-disease "
        "cohort to our analysis. We're expecting approval by May 15, 2026. This will add roughly 2,200 additional "
        "trios to the validation cohort and unlock the NHS-linked phenotype data Aisha has been waiting on."
    ),
    (
        "ASHG 2026 acceptance came in — we got a platform talk, October 14, 2026, in the 'Computational Methods "
        "in Human Genetics' session. 12-minute slot. Sarah and Javier will be co-authors on the talk; I'm the "
        "presenter."
    ),
    (
        "Public code release plan — github.com/oracle-genomics/chromatlas-nd, scheduled to go public July 2026 "
        "alongside the Nature Genetics paper. Apache 2.0 license. We'll include the Nextflow pipeline, the "
        "trained XGBoost model checkpoints, and a synthetic test dataset for reproducibility."
    ),
    (
        "Found a bug in DeepVariant 1.9 — it over-calls indels in homopolymer runs longer than 8bp, especially in "
        "Nanopore data. We've patched it locally and are retraining the indel model on our internal training set. "
        "Reported upstream to Google as DeepVariant issue #2147."
    ),
    (
        "Joining the federated network: Dr. Hiroshi Tanaka at RIKEN Yokohama is bringing in a Japanese rare-disease "
        "cohort, approximately 400 trios, primarily East Asian ancestry. This addresses one part of R1's ancestry "
        "diversification concern from the Nature Genetics review."
    ),
    (
        "Recall check — when is the IRB amendment expected to be approved, and when is my ASHG talk scheduled?"
    ),
    (
        "Headline pilot result — in our 850-case unsolved DDD pilot, 12% of cases received a candidate non-coding "
        "variant from our pipeline, and 4% of the original 850 (so a third of the candidates) were independently "
        "confirmed by functional assay. That's the diagnostic yield gain we're going to lead with in the paper."
    ),
    (
        "On the functional assay side: we run luciferase reporter assays in HEK293 cells as the high-throughput "
        "screen, then validate hits in iPSC-derived cortical neurons through Sarah's lab at Baylor. The HEK293 "
        "screen runs on a Hamilton Star liquid handler, ~96 candidates per week."
    ),
    (
        "Another patient case to remember: ND-3017, a 4-year-old with severe intellectual disability and absent "
        "speech. Our pipeline identified a 47bp insertion in a fetal-cortex-specific enhancer of FOXP2. The "
        "luciferase reporter assay showed 6.2-fold enhancer activity loss. Submitted as a candidate diagnosis."
    ),
    (
        "Software discipline update — we now enforce ruff and mypy --strict via pre-commit hooks on the analysis "
        "repo, no exceptions. CI fails the PR if either fails. This came after the May incident where a typing "
        "error in the variant-merging code corrupted three weeks of trio outputs."
    ),
    (
        "Recall check — in the 850-case unsolved DDD pilot, what proportion of cases received a candidate variant, "
        "and what proportion were functionally validated?"
    ),
    (
        "Funding update — NIH approved a supplement for ancestry expansion, +680K for FY2026. This funds two more "
        "postdocs and additional sequencing capacity for under-represented populations. We need to spend it by "
        "September 2026 or it lapses."
    ),
    (
        "Cloud vendor switch on the staging side — moving from AWS S3 to OCI Object Storage for the intermediate "
        "FASTQ and BAM staging. Cost reduction of about 40 percent at our volume. The clinical-validation enclave "
        "stays on the HIPAA-audited Oracle Cloud Ashburn region as before."
    ),
    (
        "Adding a new replication cohort — the SFARI Simons Simplex Collection, 1,200 autism trios, will be used "
        "for replication of the autism subset of our findings. SFARI Base access approved last week; data transfer "
        "starts mid-May 2026."
    ),
    (
        "Found a data leakage problem — six samples from Sarah's CUT&Tag dataset overlap with our Enformer "
        "fine-tuning training set. We've fixed it by stratifying splits at the donor level rather than the sample "
        "level. Re-running the affected models this week; results should not change materially."
    ),
    (
        "Pipeline architecture for the record — 12 stages from raw FASTQ to clinical report: QC, alignment, joint "
        "calling, SV calling, variant annotation, Enformer scoring, hybrid model scoring, structural impact, "
        "splicing analysis, phenotype matching, ranking, report generation. Average wall-time is 14 hours per trio "
        "on the new H100 pod."
    ),
    (
        "Recall check — which of my collaborators are based on the US East Coast specifically, and what new "
        "replication cohort did we add?"
    ),
    (
        "Steering committee meeting on May 1, 2026 just wrapped — committee approved phase 2 of the clinical "
        "validation, expanding from 850 to 2,350 cases, total budget +1.2M from existing grant lines. Also approved "
        "expansion of the Texas Children's GenomeBot deployment to two additional pediatric hospitals."
    ),
    (
        "Manuscript update — Nature Genetics editor suggested merging the methods paper into the main atlas paper "
        "as one combined submission. We agreed. Single submission now targeting July 15, 2026 with both atlas and "
        "methods content. The clinical impact paper still goes to AJHG separately in 2027."
    ),
    (
        "Algorithm change — replaced XGBoost with LightGBM for the hybrid prioritization model. 3x training "
        "speedup, AUROC unchanged at 0.91 on the ClinVar holdout. This makes our hyperparameter sweeps more "
        "tractable on the supplement-funded compute budget."
    ),
    (
        "Recall check — why did we switch from XGBoost to LightGBM, and did the AUROC change?"
    ),
    # ---------- Block 4: additional declarative + recall (turns 62-80) ----------
    (
        "ASHG planning — Sarah and I are co-presenting a 90-minute workshop on multi-modal genomic agent memory "
        "the day before the main meeting, October 13, 2026. Expecting around 150 attendees; the demo will use "
        "our actual ChromAtlas-Viewer streamlit app on synthetic patient data."
    ),
    (
        "Hire #2 — Eleni Markou is starting April 8, 2026 as a senior software engineer. Background in production "
        "ML systems at Stripe. Her focus will be hardening the production VEP pipeline that GenomeBot uses at "
        "Texas Children's, with explicit SLOs and on-call rotation."
    ),
    (
        "HIPAA update — we passed our annual HIPAA audit this month with zero findings. Certification valid through "
        "Q2 2027. The auditor specifically noted our differential-privacy federated training as exemplary practice."
    ),
    (
        "Patient case ND-4022 — 8-year-old with autism plus drug-resistant epilepsy. Our pipeline flagged a 250bp "
        "deletion in a long-range enhancer of CNTNAP2, ~1.2 megabases distal to the gene body, looped via Hi-C "
        "in fetal cortex. Currently under reporter assay; results expected in two weeks."
    ),
    (
        "Recall check — who are the two new hires we've added this year, when are/were their start dates, and "
        "what does each one focus on?"
    ),
    (
        "Updated publication plan — clinical impact paper now planned for AJHG submission January 2027, with "
        "two-year follow-up data on the validation cohort. We need that extra time window to capture downstream "
        "clinical action: changes in management, new therapies started, etc."
    ),
    (
        "R1 reviewer feedback addressed — added 540 trios from the 1000 Genomes Project AFR superpopulation and "
        "began a Hispanic/Latino expansion via the All of Us Research Program. Final ancestry composition for "
        "the resubmission: 41% EUR, 22% AFR, 18% EAS, 12% AMR, 7% SAS."
    ),
    (
        "Tool we built — ChromAtlas-Viewer, a streamlit app that lets clinicians inspect candidate variants "
        "visually with linked Hi-C contacts, Enformer attribution heatmaps, and ortholog conservation tracks. "
        "Deployed at Texas Children's; about 80 active monthly users so far."
    ),
    (
        "Performance target — current pipeline classifies a trio in 14 hours on the H100 pod; we are targeting "
        "6 hours by Q4 2026 via mixed-precision Enformer inference and asynchronous DAG scheduling. This will "
        "let us hit the SLA for emergent clinical cases at TCH."
    ),
    (
        "Recall check — what is our current per-trio classification time and what is the Q4 2026 target?"
    ),
    (
        "Strategic update — Oracle Health partnership is expanding. Starting Q3 2026, ChromAtlas-ND will be the "
        "reference implementation for ECRI's clinical genomics offering, which means we'll be the upstream that "
        "feeds variant prioritization for ECRI's clinical decision support across their hospital network."
    ),
    (
        "Personnel change — Aisha Patel is returning to her substantive role at Genomics England in July 2026 "
        "after her two-year visiting stint with us. Her postdoc, Dr. Priya Raghavan, becomes the new lead on the "
        "PRS validation arm and stays embedded with our team through 2027."
    ),
    (
        "Bug fix — Pangolin 1.5 was reporting false-positive splice gains for variants in introns longer than "
        "20kb. Patched with an intron-length-aware threshold that scales the splice-gain probability cap. Pull "
        "request submitted upstream; merged last Tuesday."
    ),
    (
        "Validation cohort milestone — 425 of the 850 cases have been sequenced and run through the pipeline so "
        "far. 38 candidate diagnoses produced. 11 of those have been confirmed by Sanger sequencing in CLIA-"
        "certified labs. On track for the planned interim analysis at 600 cases."
    ),
    (
        "Recall check — who is replacing Aisha as the PRS validation lead, and which institution is Aisha "
        "returning to?"
    ),
    (
        "Next steering committee meeting is August 1, 2026. Agenda includes the phase 2 expansion progress, "
        "the cost forecast through end of grant period, and a discussion of whether to seek a P01 program project "
        "renewal versus going for a U01 cooperative agreement."
    ),
    (
        "Recall check — what was the outcome of the most recent steering committee meeting, what was approved, "
        "and how does it affect the cohort size?"
    ),
    (
        "Synthesis recall — give me a complete personnel update covering every change since the project started: "
        "all collaborators, all hires, all departures, and current role assignments."
    ),
    (
        "Final synthesis — give me a complete briefing covering: the cohort and sequencing platforms, all "
        "collaborators across all institutions, the three analytical strategies plus the LightGBM switch, both "
        "new hires this year, the patient cases ND-2104, ND-3017, and ND-4022 with their respective variants, "
        "all major tool changes (DELLY, LightGBM, Pangolin patch), the funding supplement and ancestry expansion, "
        "the upcoming August 2026 steering meeting, and the Q3 2026 Oracle Health partnership expansion."
    ),
]

FULL_CONVERSATION_TURNS = list(conversation_turns)
if MAX_BENCHMARK_TURNS > 0:
    conversation_turns = conversation_turns[:MAX_BENCHMARK_TURNS]
print(f"Benchmark conversation turns: {len(conversation_turns)} of {len(FULL_CONVERSATION_TURNS)}")
print(f"Running {len(conversation_turns)} turns through both agents...\n")

for i, q in enumerate(conversation_turns, 1):
    print(f"=== Turn {i:2d}/{len(conversation_turns)} ===")
    print(f"USER: {q[:140]}{'...' if len(q) > 140 else ''}")

    oamp_answer = call_oamp_agent(q)
    naive_answer = call_naive_agent(q)

    print(f"  [OAMP,  {oamp_token_history[-1]:>6} tok, {oamp_total_latency[-1]:5.1f}s] {oamp_answer[:140]}")
    print(f"  [Naive, {naive_token_history[-1]:>6} tok, {naive_total_latency[-1]:5.1f}s] {naive_answer[:140]}")
    print()



---

## Benchmark 1 — Token Consumption

How many input tokens does each agent send to the model, turn after turn?


In [ ]:
if "align_metric_series" not in globals():
    def align_metric_series(label: str, *series):
        lengths = [len(values) for values in series]
        if not lengths or min(lengths) == 0:
            raise RuntimeError(f"No data to plot for {label}. Run the benchmark cells that populate the metric arrays first.")
        n = min(lengths)
        if len(set(lengths)) != 1:
            print(f"[warn] {label}: aligning series to the first {n} turns; observed lengths={lengths}.")
            print("       Restart the kernel and rerun from the setup cells for a clean full comparison.")
        return list(range(1, n + 1)), [list(values)[:n] for values in series]

turns, (oamp_tokens, naive_tokens) = align_metric_series(
    "OAMP vs naive token plot",
    oamp_token_history,
    naive_token_history,
)

plt.figure(figsize=(10, 5))
plt.plot(turns, naive_tokens, marker="s", color="#F44336",
         label="Naive memory (flat accumulated history)", linewidth=2)
plt.plot(turns, oamp_tokens, marker="o", color="#4CAF50",
         label="OAMP memory (retrieved context card)", linewidth=2)
plt.xlabel("Conversation Turn")
plt.ylabel("Estimated Input Tokens per Request")
plt.title("Token Consumption: OAMP vs Naive Memory")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

total_naive = sum(naive_tokens)
total_oamp = sum(oamp_tokens)
savings = (1 - total_oamp / total_naive) * 100 if total_naive else 0.0

print("\n" + "=" * 60)
print(f"{'Turn':<6}{'OAMP tokens':>14}{'Naive tokens':>16}{'Growth ratio':>18}")
print("=" * 60)
for t, o, n in zip(turns, oamp_tokens, naive_tokens):
    ratio = n / o if o else float("inf")
    print(f"{t:<6}{o:>14,}{n:>16,}{ratio:>17.1f}x")
print("=" * 60)
print(f"{'TOTAL':<6}{total_oamp:>14,}{total_naive:>16,}")
print(f"\nOAMP sent {savings:.1f}% fewer input tokens across the plotted turns.")
print(f"By turn {turns[-1]}, the naive agent is sending {naive_tokens[-1] / oamp_tokens[-1]:.1f}x "
      f"more tokens per request than the OAMP agent.")


---

## Benchmark 2 — Latency

Does OAMP's smaller prompt win enough time in the model call to offset its own retrieval overhead? Each turn is split into **retrieval latency** (prompt assembly) and **end-to-end latency** (query → answer).


In [ ]:
if "align_metric_series" not in globals():
    def align_metric_series(label: str, *series):
        lengths = [len(values) for values in series]
        if not lengths or min(lengths) == 0:
            raise RuntimeError(f"No data to plot for {label}. Run the benchmark cells that populate the metric arrays first.")
        n = min(lengths)
        if len(set(lengths)) != 1:
            print(f"[warn] {label}: aligning series to the first {n} turns; observed lengths={lengths}.")
            print("       Restart the kernel and rerun from the setup cells for a clean full comparison.")
        return list(range(1, n + 1)), [list(values)[:n] for values in series]

import numpy as np

turns, (oamp_total, naive_total, oamp_retrieval, naive_retrieval) = align_metric_series(
    "OAMP vs naive latency plot",
    oamp_total_latency,
    naive_total_latency,
    oamp_retrieval_latency,
    naive_retrieval_latency,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(turns, naive_total, marker="s", color="#F44336",
         label="Naive end-to-end", linewidth=2, markersize=4)
ax1.plot(turns, oamp_total, marker="o", color="#4CAF50",
         label="OAMP end-to-end", linewidth=2, markersize=4)
ax1.set_xlabel("Conversation Turn")
ax1.set_ylabel("Seconds (log scale)")
ax1.set_title("End-to-End Latency per Turn (log scale; robust to outliers)")
ax1.set_yscale("log")
ax1.grid(True, alpha=0.3, which="both")
ax1.legend()

ax2.plot(turns, oamp_retrieval, marker="o", color="#4CAF50",
         label="OAMP retrieval (add+context_card)", linewidth=2, markersize=4)
ax2.plot(turns, naive_retrieval, marker="s", color="#F44336",
         label="Naive retrieval (list append)", linewidth=2, markersize=4)
ax2.set_xlabel("Conversation Turn")
ax2.set_ylabel("Seconds (log scale)")
ax2.set_title("Retrieval Latency per Turn (log scale)")
ax2.set_yscale("log")
ax2.grid(True, alpha=0.3, which="both")
ax2.legend()

plt.tight_layout()
plt.show()


def _pcts(arr):
    a = np.asarray(arr, dtype=float)
    return float(np.mean(a)), float(np.median(a)), float(np.percentile(a, 95)), float(np.max(a))


print("\n" + "=" * 90)
print(f"{'Metric':<30}{'mean':>12}{'p50':>12}{'p95':>12}{'max':>12}{'where':>12}")
print("=" * 90)

for label, arr in [
    ("OAMP end-to-end (s)", oamp_total),
    ("Naive end-to-end (s)", naive_total),
    ("OAMP retrieval (s)", oamp_retrieval),
    ("Naive retrieval (s)", naive_retrieval),
]:
    mean, p50, p95, mx = _pcts(arr)
    where = int(np.argmax(arr)) + 1
    print(f"{label:<30}{mean:>11.2f}s{p50:>11.2f}s{p95:>11.2f}s{mx:>11.2f}s{('turn ' + str(where)):>12}")

print("=" * 90)
print("\nMean is sensitive to tail-latency outliers; p50 (median) and p95 give a more honest view of typical performance.")


> **Why doesn't naive latency grow with conversation length?**
>
> OCI Generative AI's API performs **automatic prompt caching** on repeated request prefixes. The naive agent's `messages` list is append-only — every turn starts with the exact same bytes as the previous turn, then adds a new tail. The cached prefix is essentially free to re-process, so input prefill stays roughly constant in wall-clock time even as the prompt grows to tens of thousands of tokens. End-to-end latency for the naive agent is then dominated by **output decoding** (≈50-150 tokens for 1-3 sentence answers), which is why that line sits around 2 seconds.
>
> **Why is OAMP slower here?**
>
> 1. The OAMP agent rebuilds its prompt every turn as `system + retrieved context_card + latest user message`. Because `context_card` changes from turn to turn, there is **no stable prefix** to cache — every call pays full prefill cost.
> 2. With `extract_memories=True` and a flagship LLM attached, every `add_messages()` call triggers extra `xai.grok-3-fast` round-trips for memory extraction and summary updates. Those extra calls are most of the 4-8 second "retrieval latency" you see on the right-hand chart.
>
> Under short answers and a stable cached prefix, naive latency is dominated by output decoding and stays flat. The token-cost win for OAMP (Benchmark 1) is real and persistent, but the *latency* win only appears when caching breaks down — long sessions that exceed the cache window, output-heavy answers, or restarts across days.
>
> ---
>
> **Cache-friendly patterns for an OAMP-backed agent**
>
> If you want OAMP's structured memory **and** the latency benefit of prompt caching, structure the prompt so most of it is byte-stable across turns:
>
> 1. **Stable prefix, small dynamic tail.** Put the system prompt + a slowly-changing "core memory" block at the top of every prompt. Append the freshly-retrieved context card and the latest user message at the end. The first ≥1024 cached tokens are what matters — the small dynamic tail costs almost nothing.
> 2. **Keep the system prompt byte-identical every turn.** Even a rotating timestamp or session ID in the system prompt defeats caching. Put volatile values *after* the cacheable prefix.
> 3. **Mirror the conversation history alongside OAMP.** Keep an append-only `messages` list as your prompt source (so OCI Generative AI caches it) and use OAMP only to **enrich** the latest turn with retrieved facts. You give back the per-turn token-reduction benefit, but you keep durable cross-session memory plus consistent low latency.
> 4. **Use a small model for extraction.** `OciSdkLlm` for extraction, ideally pointed at a lower-cost OCI model when one is available, while keeping `xai.grok-3-fast` for the user-facing reply. Extraction is mechanical and doesn't need the flagship.
> 5. **Lower extraction frequency.** `memory_extraction_frequency=10` fires every 10 messages instead of every 2, cutting extraction calls roughly 5×.
> 6. **Move extraction off the hot path.** Set `extract_memories=False` on the client and run a background/batch job that calls the extractor when the user is not waiting. This eliminates extraction latency from the user-perceived response time entirely.
>
> Combine (1) + (3) + (6) and you typically get: same memory quality, output-decode-bound latency similar to the naive agent, and OAMP's cross-session durability still intact.


> ---
>
> **Why does OAMP basic occasionally spike to multiple minutes on one turn?**
>
> The OAMP basic agent's `add_messages()` makes two extra `xai.grok-3-fast` calls (memory extraction + summary refresh) that neither the naive nor the cache-friendly agent has on the hot path. Those calls are subject to **flagship-model tail latency** — a real and well-known phenomenon where a single request occasionally takes minutes instead of seconds, due to API congestion, transient rate-limit retries, or genuinely slow generation on a long input.
>
> When it happens, it usually shows up at the **last turn** of a long conversation, where (a) the input is densest (synthesis prompts) and (b) the running summary has accumulated state to roll forward. The chart above is on a log y-axis specifically so a single 10-minute outlier doesn't compress every other point into a flat line.
>
> **Look at p50 and p95, not just mean.** Mean is dominated by tail outliers. p50 and p95 in the table above give the honest "typical" picture.
>
> **Production fix**: use a small model (`xai.grok-3-fast` / `xai.grok-3-fast-nano`) for the extractor and lower the extraction frequency. Or move extraction off the hot path entirely — that's the cache-friendly pattern in Benchmark 4.

---

## Benchmark 3 — Response Quality via LLM-as-a-Judge

Efficiency is worthless if the answers are wrong. The naive agent has the entire verbatim conversation in its context; in theory it should be hard to beat on recall. The OAMP agent only sees a retrieved context card. Does that cost it answer quality?

A separate `xai.grok-3-fast` call acts as an impartial judge for each turn, returning `{"winner": "A" | "B" | "Tie", "reason": "..."}`. We then plot a running cumulative score and a final total-wins bar chart.


In [ ]:
JUDGE_PROMPT = """You are an impartial judge evaluating two AI assistant responses to a user query.

**User query:**
{query}

**Response A:**
{response_a}

**Response B:**
{response_b}

Evaluate both responses on:
1. Accuracy — factual correctness against the query and context mentioned earlier
2. Completeness — does it fully address what was asked
3. Relevance — on-topic and appropriately detailed
4. Coherence — well-structured and clear

Return EXACTLY this JSON object and nothing else:
{{"winner": "A" or "B" or "Tie", "reason": "one short sentence explanation"}}"""


def judge_turn(query: str, response_a: str, response_b: str) -> dict:
    raw = call_chat(
        [{
            "role": "user",
            "content": JUDGE_PROMPT.format(query=query, response_a=response_a, response_b=response_b),
        }],
        temperature=0.0,
        max_tokens=500,
    ).strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    try:
        return _json_lib.loads(raw)
    except Exception:
        return {"winner": "Tie", "reason": f"(judge parse error: {raw[:80]})"}


judgments = []
print(f"Judging {len(conversation_turns)} turn-pairs with {LLM_MODEL} as the judge...\n")
for i, (q, a, b) in enumerate(zip(conversation_turns, oamp_responses, naive_responses), 1):
    verdict = judge_turn(q, a, b)
    verdict["query"] = q
    judgments.append(verdict)
    label = {"A": "OAMP ", "B": "Naive", "Tie": "Tie  "}.get(verdict["winner"], verdict["winner"])
    print(f"Turn {i:2d}: winner={label}  {verdict['reason'][:100]}")


In [ ]:
wins = {"OAMP": 0, "Naive": 0, "Tie": 0}
oamp_running, naive_running, tie_running = [], [], []

for j in judgments:
    if j["winner"] == "A":
        wins["OAMP"] += 1
    elif j["winner"] == "B":
        wins["Naive"] += 1
    else:
        wins["Tie"] += 1
    oamp_running.append(wins["OAMP"])
    naive_running.append(wins["Naive"])
    tie_running.append(wins["Tie"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Running cumulative wins
turns = list(range(1, len(judgments) + 1))
ax1.plot(turns, oamp_running,  marker="o", color="#4CAF50", label="OAMP wins",  linewidth=2)
ax1.plot(turns, naive_running, marker="s", color="#F44336", label="Naive wins", linewidth=2)
ax1.plot(turns, tie_running,   marker="^", color="#9E9E9E", label="Ties",       linewidth=2)
ax1.set_xlabel("Conversation Turn")
ax1.set_ylabel("Cumulative Count")
ax1.set_title("Running Tally: OAMP vs Naive (judged by OCI Generative AI)")
ax1.grid(True, alpha=0.3)
ax1.legend()

# Final bar chart
colors = {"OAMP": "#4CAF50", "Naive": "#F44336", "Tie": "#9E9E9E"}
bars = ax2.bar(list(wins.keys()), list(wins.values()), color=[colors[k] for k in wins])
ax2.set_ylabel("Turns Won")
ax2.set_title(f"Final Score — OAMP {wins['OAMP']} / Naive {wins['Naive']} / Tie {wins['Tie']}")
for bar, count in zip(bars, wins.values()):
    if count > 0:
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 str(count), ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

total = len(judgments)
print(f"\n=== Final tally over {total} turns ===")
print(f"  OAMP  wins: {wins['OAMP']:3d} ({wins['OAMP']/total*100:5.1f}%)")
print(f"  Naive wins: {wins['Naive']:3d} ({wins['Naive']/total*100:5.1f}%)")
print(f"  Ties:       {wins['Tie']:3d} ({wins['Tie']/total*100:5.1f}%)")


---

## Benchmark 4 — Cache-Friendly OAMP Agent (Strategies 1 + 3 + 6)

We've now seen the trade-off:

| Agent | Tokens | Latency | Cross-session memory |
|---|---|---|---|
| Naive | grows linearly (expensive) | low — cached prefix + short outputs | none |
| OAMP basic | flat (cheap) | high — extraction LLM calls every turn | yes |

**The goal of this section**: build a third agent that gets **OAMP's durable memory** *and* **the naive agent's low latency**. We accept giving back the per-turn token-reduction win to recover prompt caching.

### The three strategies, structured

| # | Strategy | Why it matters | How we implement it |
|---|---|---|---|
| **1** | **Stable prefix + small dynamic tail** | OCI Generative AI's automatic prompt cache only matches when the *byte-identical* prefix repeats across requests. Any rebuild of earlier content kills the cache and forces full prefill. | Keep the system prompt and earlier conversation byte-stable across turns. Append retrieved-memory hints to the END of the latest user message, never at the front. |
| **3** | **Mirrored append-only history alongside OAMP** | Caching requires a growing in-memory list. Durability across sessions requires a database. Do both — they serve different jobs. | An in-memory `cached_messages` list is the *prompt source* (so caching works). Every turn is also persisted to an OAMP thread (so memory survives the process). |
| **6** | **Offline extraction, off the hot path** | In the basic OAMP agent, every `add_messages()` call triggered a flagship `xai.grok-3-fast` extraction call. That dominated user-visible latency. The user shouldn't wait for memory housekeeping. | Build the OAMP client with `extract_memories=False`. After the conversation (or in a background worker), run one batch extraction call against the transcript using a *small* model — we use `xai.grok-3-fast`. |

The full implementation follows.


### Adding a fourth optimization: periodic compaction

The three strategies above keep latency low and memory durable, but per-turn token cost still grows linearly because we're sending the full history. For long conversations, we add a **fourth layer**:

- **Compact older messages every N turns.** Every 20 turns, summarise everything older than the last 4 turns into a single system-message block, replace the older messages in the in-memory list with that summary, and continue. The model still sees full fidelity for the most recent turns. We use a *small* model (`xai.grok-3-fast`) for the summary call so the compaction itself is cheap.

This is a deliberate, scheduled cache reset. Token growth becomes a **sawtooth** rather than a straight line, and end-to-end latency stays bounded — we accept one cold-prefix turn every 20 turns in exchange for that.

| Cache-friendly OAMP, compaction every 20 turns | Result |
|---|---|
| Turns 1-19 | Linear token growth, prefix cache hits |
| Turn 20 | Compaction: messages list collapses → system + summary + last 4 turns |
| Turns 21-39 | Linear growth resumes from a small base |
| Turn 40 | Compaction again: rolling summary updates |
| ... | sawtooth pattern repeats |


In [ ]:
import time

cached_connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)

cached_client = OracleAgentMemory(
    connection=cached_connection,
    embedder=oamp_embedder,
    extract_memories=False,
    schema_policy="create_if_necessary",
)

CACHED_USER = "benchmark-user-cached"
CACHED_AGENT = "benchmark-agent-cached"

for fn, eid, info in [
    (cached_client.add_user, CACHED_USER, "Richmond - cache-friendly benchmark."),
    (cached_client.add_agent, CACHED_AGENT, "Cache-friendly OAMP-backed assistant."),
]:
    try:
        fn(eid, info)
    except ValueError as exc:
        if "already exists" not in str(exc):
            raise

cached_thread = cached_client.create_thread(user_id=CACHED_USER, agent_id=CACHED_AGENT)

cached_token_history = []
cached_retrieval_latency = []
cached_total_latency = []
cached_responses = []
cached_messages = [{"role": "system", "content": SYSTEM_PROMPT}]

COMPACTION_INTERVAL = 20
KEEP_RECENT_TURNS = 4


def call_oamp_cached_agent(user_query: str) -> str:
    """Cache-friendly OAMP agent."""
    t_start = time.perf_counter()
    t_retrieve_start = time.perf_counter()
    hits = cached_client.search(
        user_query,
        user_id=CACHED_USER,
        agent_id=CACHED_AGENT,
        max_results=3,
        record_types=["memory"],
    )
    if hits:
        hint = "\n\n[Memory hints from prior sessions:\n" + "\n".join(f"- {h.content}" for h in hits) + "]"
    else:
        hint = ""
    t_retrieve_end = time.perf_counter()

    cached_messages.append({"role": "user", "content": user_query + hint})
    cached_token_history.append(estimate_tokens(cached_messages))

    answer = call_chat(cached_messages)
    cached_messages.append({"role": "assistant", "content": answer})

    cached_thread.add_messages([
        Message(role="user", content=user_query),
        Message(role="assistant", content=answer),
    ])

    t_end = time.perf_counter()
    cached_retrieval_latency.append(t_retrieve_end - t_retrieve_start)
    cached_total_latency.append(t_end - t_start)
    cached_responses.append(answer)
    return answer


def compact_older_messages(turn_number: int) -> bool:
    """Replace older messages with a model-written summary every COMPACTION_INTERVAL turns."""
    if turn_number == 0 or turn_number % COMPACTION_INTERVAL != 0:
        return False

    keep_msg_count = 2 * KEEP_RECENT_TURNS
    if len(cached_messages) <= 1 + keep_msg_count + 2:
        return False

    older = cached_messages[1:-keep_msg_count]
    recent = cached_messages[-keep_msg_count:]
    if not older:
        return False

    transcript = "\n".join(f"[{m['role']}] {m['content']}" for m in older)
    summary_text = call_chat(
        [{
            "role": "user",
            "content": (
                "Condense this conversation history into a concise summary. Preserve ALL named entities, numbers, dates, decisions, "
                "tool/method names, patient case identifiers, and personnel changes. Aim for about 30-40% of the original length.\n\n"
                f"Transcript:\n{transcript}"
            ),
        }],
        temperature=0.1,
        max_tokens=1600,
    ).strip()

    cached_messages[:] = (
        [cached_messages[0]]
        + [{"role": "system", "content": f"[Earlier conversation summary through turn {turn_number - KEEP_RECENT_TURNS}]\n{summary_text}"}]
        + recent
    )
    return True


print("Cache-friendly OAMP agent ready.")
print(f"Compaction every {COMPACTION_INTERVAL} turns, keeping last {KEEP_RECENT_TURNS} turns verbatim.")


### Why this design

Walking through the implementation top-to-bottom:

**The client (`extract_memories=False`)** — we deliberately don't attach an `Llm`. Without that, `add_messages()` is just a database insert. No extraction, no summary update, no flagship-model round-trip on the hot path. This single change removes the dominant source of latency in the basic OAMP agent.

**The retrieval (`cached_client.search(...)` → tail)** — we still query OAMP for relevant memories, but we *append* them to the latest user message rather than constructing a brand-new prompt. Everything before that latest turn is byte-stable from the previous request, so OCI Generative AI's automatic prefix cache hits. The retrieved hint is short (a few facts), so the cache *miss* on the new tail is small.

**The append-only `cached_messages` list (Strategy 1)** — this is the prompt source. We must never edit earlier entries (no rewrites, no inserts in the middle), or the cache breaks for every subsequent turn. Append only.

**The mirrored persistence (Strategy 3)** — `cached_thread.add_messages([...])` writes the turn to Oracle so future sessions can recover it via `get_thread(thread_id)`. The in-memory list is the *prompt source*; the OAMP thread is the *durable store*. They stay in lock-step but serve different jobs.

**Offline extraction (Strategy 6)** — done after the conversation in one batch call to a small model. We'll run that step a few cells down so you can see it.


### Run the same turns through the cache-friendly agent

We reuse the `conversation_turns` list from Section 4. The output table will look almost identical to the naive agent's — same answers, similar token counts — but the latency story will be different.


In [ ]:
print(f"Running {len(conversation_turns)} turns through the cache-friendly OAMP agent...\n")
for i, q in enumerate(conversation_turns, 1):
    print(f"=== Turn {i:2d}/{len(conversation_turns)} ===")
    print(f"USER: {q[:140]}{'...' if len(q) > 140 else ''}")
    answer = call_oamp_cached_agent(q)
    print(f"  [Cached, {cached_token_history[-1]:>6} tok, {cached_total_latency[-1]:5.1f}s] {answer[:140]}")
    if i % COMPACTION_INTERVAL == 0:
        before_len = len(cached_messages)
        if compact_older_messages(i):
            print(f"  [Compacted at turn {i}: {before_len} messages → {len(cached_messages)} messages — cache reset at this boundary]")
    print()


### Compare all three patterns on token consumption

The cache-friendly agent grows linearly like the naive agent (it sends the full history). The basic OAMP agent stays flat. This is the trade we made — accept token growth in exchange for durable memory + low latency.


In [ ]:
if "align_metric_series" not in globals():
    def align_metric_series(label: str, *series):
        lengths = [len(values) for values in series]
        if not lengths or min(lengths) == 0:
            raise RuntimeError(f"No data to plot for {label}. Run the benchmark cells that populate the metric arrays first.")
        n = min(lengths)
        if len(set(lengths)) != 1:
            print(f"[warn] {label}: aligning series to the first {n} turns; observed lengths={lengths}.")
            print("       Restart the kernel and rerun from the setup cells for a clean full comparison.")
        return list(range(1, n + 1)), [list(values)[:n] for values in series]

turns, (naive_tokens, oamp_tokens, cached_tokens) = align_metric_series(
    "three-pattern token plot",
    naive_token_history,
    oamp_token_history,
    cached_token_history,
)

plt.figure(figsize=(10, 5))
plt.plot(turns, naive_tokens, marker="s", color="#F44336", linewidth=2, label="Naive (full history)")
plt.plot(turns, oamp_tokens, marker="o", color="#4CAF50", linewidth=2, label="OAMP basic (context_card)")
plt.plot(turns, cached_tokens, marker="^", color="#2196F3", linewidth=2, label="OAMP cached (history + memory tail)")
plt.xlabel("Conversation Turn")
plt.ylabel("Estimated Input Tokens per Request")
plt.title("Token Consumption — Three Patterns")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nFinal plotted-turn input tokens:")
print(f"  Naive:       {naive_tokens[-1]:>6,}")
print(f"  OAMP basic:  {oamp_tokens[-1]:>6,}")
print(f"  OAMP cached: {cached_tokens[-1]:>6,}")


> **"OAMP cached and naive look identical on tokens — what's the benefit?"**
>
> Yes, that's the deliberate trade. Token reduction was the **basic OAMP** pattern's win. The cache-friendly pattern *gives that back* to recover prompt caching and remove extraction latency. You can't have all three of low tokens, low latency, and durable memory in a single agent — pick any two.
>
> What the cache-friendly agent provides that the naive agent **cannot**, even at identical per-turn token counts:
>
> | Dimension | Naive | OAMP cached |
> |---|---|---|
> | Cross-session memory | lost when process exits | survives forever in Oracle |
> | Cross-thread recall | none (list is per-process) | search returns facts from any prior thread |
> | Multi-instance / multi-replica deployments | each replica has its own RAM list | all replicas share the Oracle store |
> | Searchable memory | linear scan only | vector search over all stored memories |
> | Multi-user isolation | manual scoping in app code | first-class `user_id` / `agent_id` scoping |
> | Auditability | none | timestamps, record types, ownership tracked in DB |
>
> Three-way summary:
>
> | | Token cost | Latency | Durable memory |
> |---|---|---|---|
> | Naive | linear | low | **none** |
> | OAMP basic | flat | high | yes |
> | OAMP cached | linear | low | yes |
>
> **Choose by your dominant constraint**:
> - Per-turn token cost dominates → use **OAMP basic** (`get_context_card`, accept higher latency)
> - User-perceived latency dominates and you need durable memory → use **OAMP cached** (this section)
> - Short-lived, single-process, no continuity needed → **naive** is fine
>
> If you eventually need *all three* (low tokens + low latency + durable memory), the next step is **periodic summarisation**: every N turns (e.g. 50–100), call `thread.get_summary()` to compact older messages, replace the older entries in your in-memory list with the summary, and accept *one* cache reset at the boundary. That layered pattern is on top of OAMP cached, not a replacement for it.

>
> **Update with compaction enabled (Benchmark 4 below):** the cache-friendly agent now compacts older messages every 20 turns, so its token line is a *sawtooth* — linear growth between compaction points, then a drop. Over 80 turns this keeps the cached agent's average prompt size meaningfully below the naive agent's, while still preserving caching within each 20-turn window.

### Compare all three patterns on latency

This is the headline chart. The cache-friendly agent should track the naive agent closely on end-to-end latency (both benefit from prompt caching and have no extraction overhead). The basic OAMP agent's extra LLM round-trips per turn are the gap.


In [ ]:
if "align_metric_series" not in globals():
    def align_metric_series(label: str, *series):
        lengths = [len(values) for values in series]
        if not lengths or min(lengths) == 0:
            raise RuntimeError(f"No data to plot for {label}. Run the benchmark cells that populate the metric arrays first.")
        n = min(lengths)
        if len(set(lengths)) != 1:
            print(f"[warn] {label}: aligning series to the first {n} turns; observed lengths={lengths}.")
            print("       Restart the kernel and rerun from the setup cells for a clean full comparison.")
        return list(range(1, n + 1)), [list(values)[:n] for values in series]

turns, (
    naive_total,
    oamp_total,
    cached_total,
    naive_retrieval,
    oamp_retrieval,
    cached_retrieval,
) = align_metric_series(
    "three-pattern latency plot",
    naive_total_latency,
    oamp_total_latency,
    cached_total_latency,
    naive_retrieval_latency,
    oamp_retrieval_latency,
    cached_retrieval_latency,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(turns, naive_total, marker="s", color="#F44336", linewidth=2, markersize=4, label="Naive end-to-end")
ax1.plot(turns, oamp_total, marker="o", color="#4CAF50", linewidth=2, markersize=4, label="OAMP basic end-to-end")
ax1.plot(turns, cached_total, marker="^", color="#2196F3", linewidth=2, markersize=4, label="OAMP cached end-to-end")
ax1.set_xlabel("Conversation Turn")
ax1.set_ylabel("Seconds (log scale)")
ax1.set_title("End-to-End Latency — Three Patterns (log scale)")
ax1.set_yscale("log")
ax1.grid(True, alpha=0.3, which="both")
ax1.legend()

ax2.plot(turns, naive_retrieval, marker="s", color="#F44336", linewidth=2, markersize=4, label="Naive retrieval (list append)")
ax2.plot(turns, oamp_retrieval, marker="o", color="#4CAF50", linewidth=2, markersize=4, label="OAMP basic retrieval (add+context_card+extract)")
ax2.plot(turns, cached_retrieval, marker="^", color="#2196F3", linewidth=2, markersize=4, label="OAMP cached retrieval (search only)")
ax2.set_xlabel("Conversation Turn")
ax2.set_ylabel("Seconds (log scale)")
ax2.set_title("Retrieval Latency — Three Patterns (log scale)")
ax2.set_yscale("log")
ax2.grid(True, alpha=0.3, which="both")
ax2.legend()

plt.tight_layout()
plt.show()


def _pcts3(arr):
    a = np.asarray(arr, dtype=float)
    return (float(np.mean(a)),
            float(np.median(a)),
            float(np.percentile(a, 95)),
            float(np.max(a)),
            int(np.argmax(a)) + 1)


print("\n" + "=" * 110)
print(f"{'Metric':<32}{'pattern':>14}{'mean':>10}{'p50':>10}{'p95':>10}{'max':>10}{'max@turn':>14}")
print("=" * 110)

rows = [
    ("End-to-end per turn (s)", "Naive", naive_total),
    ("End-to-end per turn (s)", "OAMP basic", oamp_total),
    ("End-to-end per turn (s)", "OAMP cached", cached_total),
    ("Retrieval per turn (s)", "Naive", naive_retrieval),
    ("Retrieval per turn (s)", "OAMP basic", oamp_retrieval),
    ("Retrieval per turn (s)", "OAMP cached", cached_retrieval),
]
for label, pattern, arr in rows:
    mean, p50, p95, mx, where = _pcts3(arr)
    print(f"{label:<32}{pattern:>14}{mean:>9.2f}s{p50:>9.2f}s{p95:>9.2f}s{mx:>9.2f}s{('turn ' + str(where)):>14}")

print("=" * 110)
print()
print("Total wall-clock time across the plotted turns:")
print(f"  Naive:       {sum(naive_total):>7.1f}s")
print(f"  OAMP basic:  {sum(oamp_total):>7.1f}s   (sum is sensitive to tail-latency outliers)")
print(f"  OAMP cached: {sum(cached_total):>7.1f}s")


> **Reading this chart**
>
> The y-axes are **log-scale**, so a 10-minute outlier doesn't compress every other point into a flat line. Look at the **p50** and **p95** rows in the table above for the typical-case story; mean and total are sensitive to single tail-latency events.
>
> What you should see in a clean run:
>
> - **Naive** and **OAMP cached** sit at roughly the same low level on end-to-end latency (both benefit from prompt caching and have no extra LLM calls)
> - **OAMP basic** sits ~10-25× higher because every turn fires extraction + summary on `xai.grok-3-fast`
> - **OAMP cached retrieval** is just a vector search (~milliseconds), much faster than even **OAMP basic retrieval** which includes the extraction round-trip
> - Occasional spikes on the OAMP basic line are **flagship-model tail latency** on the extra LLM calls — a real production concern. The cache-friendly pattern eliminates this risk by construction (no extra LLM calls on the hot path).


### Strategy 6 in action — offline batch extraction

Now that the user is no longer waiting, we run **one** extraction call against the transcript. In production this would happen in a background worker, a queue consumer, or a nightly cron job, not on the user-facing path.

This recovers OAMP's automatic durable memory feature for the cache-friendly agent: the next time someone opens this thread, the extracted facts are queryable via `cached_client.search(...)`.


In [ ]:
print("Running offline batch extraction...")
t_extract_start = time.perf_counter()

all_messages = cached_thread.get_messages()
transcript = "\n".join(f"[{m.role}] {m.content}" for m in all_messages)

extract_prompt = f"""Extract a list of durable, atomic facts from this conversation
that would be useful for future sessions. Each fact should be self-contained
and independently meaningful.

Conversation transcript:
{transcript}

Return JSON with this exact shape: {{"facts": ["fact 1", "fact 2", ...]}}
"""

raw = call_chat(
    [{"role": "user", "content": extract_prompt}],
    temperature=0.0,
    max_tokens=2000,
).strip()
if raw.startswith("```"):
    raw = raw.strip("`")
    if raw.lower().startswith("json"):
        raw = raw[4:]
    raw = raw.strip()

try:
    facts = _json_lib.loads(raw).get("facts", [])
except Exception:
    print(f"(parse error, raw output below)\n{raw[:400]}")
    facts = []

for fact in facts:
    cached_client.add_memory(
        fact,
        user_id=CACHED_USER,
        agent_id=CACHED_AGENT,
        thread_id=cached_thread.thread_id,
    )

t_extract_end = time.perf_counter()
print(f"\nExtracted {len(facts)} durable facts in {t_extract_end - t_extract_start:.1f}s.")
print("All facts now searchable via cached_client.search() in any future session.\n")
print("First 5 extracted facts:")
for fact in facts[:5]:
    print(f"  - {fact}")


### Recommended production pattern

| Pattern | Tokens / turn | End-to-end latency | Cross-session memory | When to use |
|---|---|---|---|---|
| Naive | grows linearly | low (cached prefix) | none | short-lived sessions, no continuity needed |
| OAMP basic | flat | high (extraction on hot path) | yes | low query rate where extraction cost is acceptable, or where minimizing per-turn cost matters more than latency |
| **OAMP cached (this section)** | grows linearly | low (cached prefix) | yes | **most production agents** — durable memory + low user-perceived latency |

The cache-friendly pattern accepts that you'll spend more on input tokens (history grows) in exchange for:

- Stable, low latency dominated by output decoding
- Durable cross-session memory in Oracle
- Zero extraction cost on the user-facing critical path

When token bills become an issue at very long conversations, the next levers are layered on top of this same pattern:

- Periodically summarise older history (e.g. every 50-100 turns), replace the older messages with the summary, and accept *one* cache reset at the boundary
- Use the basic OAMP `context_card` strategy for very long-running threads where the cumulative token cost of full history exceeds the extraction cost
- Offload long tool outputs into OAMP and replace them in the prompt with short references (the offloading technique used in the basic OAMP agent)


## Cleanup

In [ ]:
try:
    client.delete_thread(thread.thread_id)
except Exception as e:
    print(f"(basic OAMP cleanup skipped: {e})")
try:
    cached_client.delete_thread(cached_thread.thread_id)
    cached_connection.close()
except NameError:
    pass  # Benchmark 4 not run
except Exception as e:
    print(f"(cached OAMP cleanup skipped: {e})")
connection.close()
print("Benchmark resources cleaned up.")
